# TT-SFUDA Model Evaluation
This notebook demonstrates how to evaluate a pretrained 2D model and visualise its predictions.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from dataset import Dataset
import archs
from metrics import iou_score
from utils import AverageMeter


In [ ]:
# Paths
dataset_name = 'hrf'  # change to your dataset
img_dir = os.path.join('TT_SFUDA_2D', 'inputs', dataset_name, 'images')
mask_dir = os.path.join('TT_SFUDA_2D', 'inputs', dataset_name, 'masks')
img_ids = [os.path.splitext(f)[0] for f in os.listdir(img_dir)]
val_set = Dataset(img_ids, img_dir, mask_dir, '.png', '.png', num_classes=1)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=1, shuffle=False)

model = archs.UNet(num_classes=1)
model.load_state_dict(torch.load('models/source_model.pth', map_location='cpu'))
model.eval();


In [ ]:
def evaluate(loader, model):
    meter = AverageMeter()
    for img, mask, _ in loader:
        with torch.no_grad():
            output = model(torch.tensor(img))
        _, dice = iou_score(output, torch.tensor(mask))
        meter.update(dice, img.shape[0])
    print('Dice score:', meter.avg)

evaluate(val_loader, model)


In [ ]:
# Visualise a prediction
img, mask, _ = val_set[0]
with torch.no_grad():
    pred = model(torch.tensor(img).unsqueeze(0)).sigmoid()[0,0].numpy()
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(img.transpose(1,2,0))
ax[0].set_title('Image')
ax[0].axis('off')
ax[1].imshow(mask[0], cmap='gray')
ax[1].set_title('Ground Truth')
ax[1].axis('off')
ax[2].imshow(pred > 0.5, cmap='gray')
ax[2].set_title('Prediction')
ax[2].axis('off')
plt.show()
